# Part 1.8 Time travel

After this notebook you can go back to an earlier checkpoint, change one fact, and run
forward on a new branch.

In [ ]:
from kea.config import setup

setup("kea-claims-local")

## See it

In Studio, open the crushed-bag run from [Part 1.3](03_claims_workflow_in_studio.ipynb).
Server running? `mise run dev`, then
https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024.

No run to open? In a new thread, attach `crushed-01.jpg` and send:

```text
Passenger P-2002, flight KA 1655. My hard-shell case was crushed. It was worth $800.
```

1. Open the thread's history and find the checkpoint after `assess_damage`.
2. Edit the state: set `assessment.severity` from 4 to 1.
3. Fork from that checkpoint and let it run.

The fork skips `human_review`, because a severity 1 crush pays less than $250. The
original run is still there, unchanged.

<img src="../../assets/screenshots/p1-08-studio-fork.png" alt="Studio's thread history with the fork action on the assessdamage checkpoint, and the edited state in the fork dialog." width="900">

<img src="../../assets/screenshots/p1-08-studio-fork-result.png" alt="the forked run's output: the lower payout and no pause at humanreview." width="900">

## Build it

The same thing in code. First a claim that pauses and is approved.

In [ ]:
from langgraph.types import Command

from kea.notebook import claim, claims_graph

graph = claims_graph()

config = {"configurable": {"thread_id": "claim-1002"}}
graph.invoke(claim("crushed-01.jpg", "pax-1002", "KA202", 800), config)
approval = {"approved": True, "payout_usd": 300, "note": "Approved at $300 after review."}
original = graph.invoke(Command(resume=approval), config)
print("original payout:", original["payout_usd"])

Go back to the checkpoint just before `apply_policy`, change the assessed severity, and
run forward.

In [ ]:
history = list(graph.get_state_history(config))
before_policy = next(s for s in history if s.next == ("apply_policy",))

edited = {**before_policy.values["assessment"], "severity": 1}
fork_config = graph.update_state(before_policy.config, {"assessment": edited})

forked = graph.invoke(None, fork_config)
print("was:", original["payout_usd"], "-> now:", forked["payout_usd"])
print("asked a human:", "__interrupt__" in forked)

## Under the hood

- `get_state_history` returns checkpoints newest first. We pick the one whose `next` is
  `("apply_policy",)`.
- `update_state` writes a **new** checkpoint that branches from that point. `invoke(None,
  fork_config)` runs forward from it. **Forks branch; they don't roll back.** The
  original run is untouched.
- Replaying re-runs `record_claim`. It writes under the flight number, so the replay
  overwrites the claim instead of duplicating it. Keep side effects idempotent.
- Use it to debug ("what if the model had seen this?") and to test a fix on a real
  failure without re-running everything before it.

Docs: [Time travel](https://docs.langchain.com/oss/python/langgraph/use-time-travel).

## Recap

- Every checkpoint is still there after a run ends.
- `update_state` on an old checkpoint creates a branch; `invoke(None, ...)` runs it.
- Studio does the same from the thread history.

**Next:** [Part 1.9 Stores](09_stores.ipynb)